# Build a CUDA-enabled llama-cpp-python wheel on Kaggle

Throwaway build kernel -- internet ENABLED here only to fetch the llama-cpp-python source and pip build deps; the resulting .whl is saved to /kaggle/working so it can be downloaded and reused (bundled as an offline dataset) in the real, internet-disabled submission kernel. Compiles ON the real target GPU (RTX Pro 6000 / Blackwell) so CMAKE_CUDA_ARCHITECTURES=native picks the correct compute capability automatically.

In [ ]:
import subprocess
print("=== nvidia-smi ===")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("=== nvcc ===")
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print(r.stdout, r.stderr)

In [ ]:
import os, subprocess, sys, glob

# find the real libcuda.so.1 (the DRIVER lib, distinct from the CUDA runtime) --
# CMake's FindCUDAToolkit needs an explicit hint for this on Kaggle's container,
# LIBRARY_PATH alone isn't enough since CMake's own find_library doesn't consult it
driver_candidates = glob.glob("/usr/local/nvidia/lib64/libcuda.so*") + \
                     glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*") + \
                     glob.glob("/usr/lib/**/libcuda.so*", recursive=True)
print("libcuda.so candidates found:", driver_candidates)

env = os.environ.copy()
cmake_args = "-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=100;120"
if driver_candidates:
    stub_dir = os.path.dirname(driver_candidates[0])
    cmake_args += f" -DCMAKE_LIBRARY_PATH={stub_dir} -DCUDA_CUDA_LIBRARY={driver_candidates[0]}"
env["CMAKE_ARGS"] = cmake_args
env["FORCE_CMAKE"] = "1"
env["LIBRARY_PATH"] = os.pathsep.join(x for x in ["/usr/local/nvidia/lib64", env.get("LIBRARY_PATH", "")] if x)
print("CMAKE_ARGS:", env["CMAKE_ARGS"])

os.makedirs("/kaggle/working/wheels", exist_ok=True)
with open("/kaggle/working/build.log", "w") as logf:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "wheel", "llama-cpp-python==0.3.35",
         "--no-binary", "llama-cpp-python", "--no-deps", "-v", "-w", "/kaggle/working/wheels"],
        env=env, stdout=logf, stderr=subprocess.STDOUT,
    )
print("returncode:", r.returncode)
print("--- last 200 lines of build.log ---")
with open("/kaggle/working/build.log") as f:
    lines = f.readlines()
print("".join(lines[-200:]))


In [ ]:
import glob
print(glob.glob("/kaggle/working/wheels/*.whl"))

In [ ]:
import glob, subprocess, sys
whls = glob.glob("/kaggle/working/wheels/llama_cpp_python*.whl")
if not whls:
    print("NO WHEEL BUILT -- see build.log above for the real error")
else:
    whl = whls[0]
    subprocess.run([sys.executable, "-m", "pip", "install", "diskcache"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", whl], check=True)
    import llama_cpp
    print("llama_cpp version:", llama_cpp.__version__)
    print("supports_gpu_offload:", llama_cpp.llama_cpp.llama_supports_gpu_offload())
